In [1]:
%env XLA_PYTHON_CLIENT_PREALLOCATE=false
%env JAX_ENABLE_X64=false

env: XLA_PYTHON_CLIENT_PREALLOCATE=false
env: JAX_ENABLE_X64=false


In [2]:
import sys
sys.path.append('../src')

In [3]:
print(sys.path)

['/home/minisemin/anaconda3/envs/pinn/lib/python311.zip', '/home/minisemin/anaconda3/envs/pinn/lib/python3.11', '/home/minisemin/anaconda3/envs/pinn/lib/python3.11/lib-dynload', '', '/home/minisemin/anaconda3/envs/pinn/lib/python3.11/site-packages', '../src']


In [4]:
# Import necessary libraries

import jax
import jax.numpy as jnp

# Configure JAX to use only the CPU
# jax.config.update('jax_platform_name', 'cpu') 

# Configure JAX to use float 32 instead of 64 to reduce GPU memory but keep precision
from jax import config
config.update("jax_enable_x64", False)  
config.update("jax_default_matmul_precision", "high")  

print('jax:', jax.__version__)      # 0.7.2
print('devices:', jax.devices())    # [CudaDevice(id=0)]

import jax.experimental.layout as _layout
if not hasattr(_layout, 'DeviceLocalLayout'):
        # Provide a minimal alias so orbax can import function annotations\n",
        _layout.DeviceLocalLayout = _layout.Layout
        print('Patched jax.experimental.layout.DeviceLocalLayout ->', getattr(_layout, 'DeviceLocalLayout'))
import os
import numpy as np
import matplotlib.pyplot as plt
from pinn.train import create_train_state, train_step_batched
from pinn.cryoet_io import load_mrc_data
import pickle
from flax.training import checkpoints
from pinn.utils import initial_loss
from tqdm.notebook import trange

jax: 0.7.2
devices: [CudaDevice(id=0)]
Patched jax.experimental.layout.DeviceLocalLayout -> <class 'jax._src.layout.Layout'>


In [5]:
# Utility function to convert GPU/JAX arrays to CPU/NumPy arrays

def to_numpy(tree):
    return jax.tree_util.tree_map(
        lambda x: np.asarray(jax.device_get(x)) if isinstance(x, (jnp.ndarray, np.ndarray)) else x,
        tree,
    )

def as_f32_scalar(x):
    # Make sure to extract as a scalar (float32) even if it's an array/DeviceArray
    return np.asarray(x, dtype=np.float32).reshape(()).item()

In [6]:
NUM_CHECKPOINTS_TO_KEEP = 1000 # Checkpoint retention count (older ones get removed)

# Set loss function weights
lambda_1 = 1000000  # Data loss weight
lambda_2 = 1000     # Physics loss weight
sdf_pretrain = "sphere"

# Initialize JAX random key
key = jax.random.PRNGKey(0)

# Create train state
state, model = create_train_state(key, lambda_1=lambda_1, lambda_2=lambda_2, sdf_pretrain=sdf_pretrain, learning_rate=1e-3)
print(f"Using lambda_1: {state.lambda_1}, lambda_2: {state.lambda_2}")

Pretraining the network using SDF...
Using lambda_1: 1000000, lambda_2: 1000


In [7]:
import mrcfile

# Load MRC data
data_name = "74_downsampled"
data_path = f"../data/Chestnut_Jove_2024_downsampled/{data_name}.mrc"

with mrcfile.open(data_path, permissive=True) as mrc:
    data = mrc.data

GRID_X = data.shape[2]
GRID_Y = data.shape[1]
GRID_Z = data.shape[0]

data = load_mrc_data(data_path, data.shape)
print(f"{data_name} loaded successfully! Shape:", data.shape)
data = jnp.asarray(data, dtype=jnp.float32)

74_downsampled loaded successfully! Shape: (90, 1140, 835)


In [8]:
# Training data
num_collocation_points = 100000
x_train = jax.random.uniform(key, (num_collocation_points, 3), minval=-1.0, maxval=1.0)

# Training loop
num_steps = 10000
save_interval = 100

checkpoint_dir = os.path.abspath(f"../outputs/logs/{data_name}/lambda_{lambda_1}_{lambda_2}")

In [9]:
# Find appropriate batch size

import jax
import jax.numpy as jnp
from pinn.utils import make_phi_apply, loss_data_batched
from pinn.model import make_phi_scalar, loss_physics_batched

def _is_oom_or_backend(e: Exception) -> bool:
    s = str(e).lower()
    keys = [
        "resource_exhausted", "out of memory", "bfc_allocator",
        "no valid config", "cudnn_status_not_supported", "cudnn_status_alloc_failed"
    ]
    return any(k in s for k in keys)


def find_max_batch_size(test_fn, start=32768, step_limit=8, tol=1024, warmup=8192):
    """
    test_fn(B): 배치 B로 한 번 실행하고 .block_until_ready()까지 해주는 함수
    """
    # 1) 워밍업(작은 B로 JIT 컴파일)
    B0 = min(start, warmup)
    test_fn(B0)

    # 2) ×2 증분 탐색
    ok = B0
    for _ in range(step_limit):
        B_next = ok * 2
        try:
            test_fn(B_next)
            ok = B_next
        except Exception as e:
            if _is_oom_or_backend(e):
                break
            raise

    # 3) 이분 탐색
    lo, hi = ok, ok * 2
    while hi - lo > tol:
        mid = (lo + hi) // 2
        try:
            test_fn(mid)
            lo = mid
        except Exception as e:
            if _is_oom_or_backend(e):
                hi = mid
            else:
                raise
    return lo


# 배치 입력 -> 1D 출력 보장하는 phi_apply가 있어야 합니다.
phi_apply = make_phi_apply(state)  # (앞서 만든 버전: ravel 반환)

# 데이터 배치 테스트 함수
def _test_data_B(B):
    val = loss_data_batched(
        phi_apply,
        data.astype(jnp.float32),
        data.shape,                 # (Z,Y,X)
        batch_size=B,
        threshold=0.8,
        weight_in=0.8,
        eps=1e-8,
    )
    val.block_until_ready()

# 물리 배치 테스트 함수
def _test_phys_B(B):
    val = loss_physics_batched(
        phi_apply,
        x_train.astype(jnp.float32),
        epsilon=0.05,
        batch_size=B,
    )
    val.block_until_ready()

best_B_data = find_max_batch_size(_test_data_B, start=65536, warmup=8192)
best_B_phys = find_max_batch_size(_test_phys_B, start=8192,  warmup=4096)

print("Max safe batch_size_data:", best_B_data)
print("Max safe batch_size_phys:", best_B_phys)

/home/minisemin/anaconda3/envs/pinn/lib/python3.11/site-packages/jax/_src/numpy/lax_numpy.py:5943: UserWarning: Explicitly requested dtype <class 'jax.numpy.int64'> requested in arange is not available, and will be truncated to dtype int32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  return _arange(start, stop=stop, step=step, dtype=dtype,


Max safe batch_size_data: 4193280
Max safe batch_size_phys: 2096128


In [10]:
import shutil

# --- keep the same schema for step 0 ---
init_d = initial_loss(state, x_train, data, data.shape)     # Get the dictionary of initial losses
init_total = init_d.get("total_loss", init_d.get("total", init_d.get("loss", 0.0)))
init_data  = init_d.get("data_loss",  init_d.get("data",  0.0))
init_phys  = init_d.get("physics_loss", init_d.get("phys", init_d.get("physics", 0.0)))

payload0 = {
    "state": to_numpy(state),
    "loss": {
        "step":         np.asarray([0], dtype=np.int64),
        "total_loss":   np.asarray([as_f32_scalar(init_total)], dtype=np.float32),
        "data_loss":    np.asarray([as_f32_scalar(init_data)],  dtype=np.float32),
        "physics_loss": np.asarray([as_f32_scalar(init_phys)],  dtype=np.float32),
    }
}

if os.path.exists(checkpoint_dir):
    print(f"Removing existing checkpoint directory: {checkpoint_dir}")
    shutil.rmtree(checkpoint_dir)
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoints.save_checkpoint(
    ckpt_dir=checkpoint_dir,
    target=payload0,
    step=0,
    overwrite=False,
    keep=NUM_CHECKPOINTS_TO_KEEP,
)

loss_history = []


Removing existing checkpoint directory: /home/minisemin/matsulab/pinn-exploration-model/outputs/logs/74_downsampled/lambda_1000000_1000


In [11]:
# Self check for the dimensions

from pinn.utils import make_phi_apply 
from pinn.model import make_phi_scalar

# 1) 모델 출력 shape
y_many = state.apply_fn(state.params, jnp.zeros((4,3), jnp.float32))
y_one  = state.apply_fn(state.params, jnp.zeros((1,3), jnp.float32))
print(y_many.shape)  # (4,)
print(y_one.shape)   # (1,)  ← OK (0-D가 아니어야 함)

# 2) phi_apply/phi_scalar
phi_apply = make_phi_apply(state)
print(phi_apply(jnp.zeros((5,3))).shape)  # (5,)
print(phi_apply(jnp.zeros((1,3))).shape)  # (1,)
phi_scalar = make_phi_scalar(phi_apply)
print(phi_scalar(jnp.zeros((3,))).shape)  # ()  ← 스칼라 OK


(4,)
(1,)
(5,)
(1,)
()


In [13]:
cur_B_data = best_B_data   # 위 탐색 결과나, 없으면 초기값 예: 262144
cur_B_phys = best_B_phys   # 없으면 초기값 예: 32768

# cur_B_data = 262144
# cur_B_phys = 1024

def _is_oom(e: Exception) -> bool:
    s = str(e).lower()
    return ("resource_exhausted" in s) or ("out of memory" in s) or ("bfc_allocator" in s)


for step in trange(num_steps):
    try:
        state, loss_val, loss_data_val, loss_phys_val = train_step_batched(
            state, x_train, data,
            grid_shape=data.shape,
            batch_size_data=cur_B_data,
            batch_size_phys=cur_B_phys,
            epsilon=0.05,
        )
    except Exception as e:
        if _is_oom(e):
            # 우선 물리 배치를 절반으로 줄여 시도 (가장 무겁기 때문)
            if cur_B_phys > 2048:
                cur_B_phys = max(2048, cur_B_phys // 2)
                print(f"[OOM] physics batch ↓ {cur_B_phys}, retrying...")
                # 즉시 재시도
                state, loss_val, loss_data_val, loss_phys_val = train_step_batched(
                    state, x_train, data,
                    grid_shape=data.shape,
                    batch_size_data=cur_B_data,
                    batch_size_phys=cur_B_phys,
                    epsilon=0.05,
                )
            else:
                # 그래도 안되면 데이터 배치도 절반으로
                cur_B_data = max(16384, cur_B_data // 2)
                print(f"[OOM] data batch ↓ {cur_B_data}, retrying...")
                state, loss_val, loss_data_val, loss_phys_val = train_step_batched(
                    state, x_train, data,
                    grid_shape=data.shape,
                    batch_size_data=cur_B_data,
                    batch_size_phys=cur_B_phys,
                    epsilon=0.05,
                )
        else:
            raise

    loss_history.append({
        "step":  step + 1,
        "total_loss":   loss_val,
        "data_loss":    loss_data_val,
        "physics_loss": loss_phys_val,
    })

    if (step + 1) % save_interval == 0:
        batched_loss = {
            "step":         np.asarray([e["step"] for e in loss_history], dtype=np.int64),
            "total_loss":   np.asarray([as_f32_scalar(e["total_loss"])   for e in loss_history], dtype=np.float32),
            "data_loss":    np.asarray([as_f32_scalar(e["data_loss"])    for e in loss_history], dtype=np.float32),
            "physics_loss": np.asarray([as_f32_scalar(e["physics_loss"]) for e in loss_history], dtype=np.float32),
        }
        payload = {"state": to_numpy(state), "loss": batched_loss}
        checkpoints.save_checkpoint(ckpt_dir=checkpoint_dir, target=payload,
                                    step=step + 1, overwrite=False,
                                    keep=NUM_CHECKPOINTS_TO_KEEP)
        print(f"Checkpoint saved at step {step+1}")
        loss_history = []

  0%|          | 0/10000 [00:00<?, ?it/s]

W1002 11:19:17.149668   11688 hlo_rematerialization.cc:3198] Can't reduce memory use below 3.97GiB (4260984489 bytes) by rematerialization; only reduced to 24.90GiB (26736547872 bytes), down from 24.95GiB (26786879712 bytes) originally
W1002 11:19:28.344806   11688 bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 24.95GiB (rounded to 26786896640)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
W1002 11:19:28.345012   11688 bfc_allocator.cc:512] **********__________________________________________________________________________________________
E1002 11:19:28.345041   11688 pjrt_stream_executor_client.cc:3314] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 26786896608 bytes. [tf-allocator-allocation-error='']


[OOM] physics batch ↓ 1048064, retrying...


W1002 11:19:29.540836   11688 hlo_rematerialization.cc:3198] Can't reduce memory use below 3.97GiB (4260984489 bytes) by rematerialization; only reduced to 24.90GiB (26736547872 bytes), down from 24.95GiB (26786879712 bytes) originally
W1002 11:19:40.595403   11688 bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 24.95GiB (rounded to 26786896128)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
W1002 11:19:40.595546   11688 bfc_allocator.cc:512] **********__________________________________________________________________________________________
E1002 11:19:40.595574   11688 pjrt_stream_executor_client.cc:3314] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 26786896096 bytes. [tf-allocator-allocation-error='']


XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 26786896096 bytes.